<a href="https://colab.research.google.com/github/hindxb/FDS/blob/main/Notebooks/Project/FDS_Optimizing_Traffic_Signals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Spatial-Temporal Traffic Signal Optimization

By Hind Alzarooni - U25102248



### Objective
Use GRU-predicted traffic speeds to estimate congestion, prioritize road segments, and recommend improved green signal timing.

---

### Optimization Flow

Predicted Speed + Segment Coordinates  
↓  
Clean and Merge Traffic + Spatial Data  
↓  
Create Segment Midpoints  
↓  
Find Nearby Road Segments  
↓  
Compute Congestion Score  
↓  
Add Temporal Congestion History  
↓  
Compute Neighbor Congestion  
↓  
Calculate Spatial-Temporal Priority Score  
↓  
Group Nearby Segments into Simulated Intersections  
↓  
Allocate Green Time Based on Priority  
↓  
Estimate Optimized Speed  
↓  
Save Optimized Signal Timing Results  

---

### Key Equations

**1. Congestion Score**

\[
C = 1 - \frac{V_{pred}}{V_{max}}
\]

Where:  
- \(C\) = congestion score  
- \(V_{pred}\) = predicted speed  
- \(V_{max}\) = reference maximum speed  

---

**2. Temporal Congestion**

\[
TC = 0.60C_t + 0.30C_{t-1} + 0.10C_{t-2}
\]

Where:  
- \(TC\) = temporal congestion  
- \(C_t\) = current congestion  
- \(C_{t-1}\), \(C_{t-2}\) = previous congestion values  

---

**3. Spatial-Temporal Priority Score**

\[
P = 0.70TC + 0.30NC
\]

Where:  
- \(P\) = priority score  
- \(TC\) = temporal congestion  
- \(NC\) = neighbor congestion  

---

**4. Base Green Time**

For multiple segments in one signal group:

\[
G_{base} = \frac{120}{n}
\]

Where:  
- \(G_{base}\) = base green time  
- \(120\) = signal cycle time in seconds  
- \(n\) = number of segments in the group  

---

**5. Recommended Green Time**

\[
G_{rec} = \frac{P_i}{\sum P} \times 120
\]

Where:  
- \(G_{rec}\) = recommended green time  
- \(P_i\) = priority score of segment \(i\)  
- \(\sum P\) = total priority score in the group  

Green time is limited between 20 and 90 seconds.

---

**6. Green Time Change**

\[
\Delta G = G_{rec} - G_{base}
\]

---

**7. Estimated Speed Improvement**

\[
SI = \max(\Delta G, 0) \times 0.12 \times P
\]

---

**8. Optimized Speed**

\[
V_{opt} = V_{pred} + SI + CB
\]

Where:  
- \(V_{opt}\) = optimized speed  
- \(SI\) = estimated speed improvement  
- \(CB\) = coordination benefit from nearby traffic conditions  

---

### Output
The final output includes predicted speed, optimized speed, speed change, congestion level, priority score, base green time, recommended green time, and spatial coordination benefit.

In [ ]:

# 0. IMPORTS + MOUNT DRIVE
# numpy: numerical operations and array handling
# pandas: data loading and preprocessing
# google.colab.drive: access files stored in Google Drive
# =========================================================
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')



# 1. FILE PATHS
# =========================================================

#file path of the predicted data
predictions_path = '/content/drive/MyDrive/datasets/gru_predictions_for_map.csv'

#file path of a dataset with all the segment information
segments_path = '/content/drive/MyDrive/datasets/chicago_segments.xlsx'

#file path to output the data in for the optimization
output_path = '/content/drive/MyDrive/datasets/optimized_signal_timing.csv'



# 2. LOAD DATA
# =========================================================

#reading from the prediction and segmentation datasets
pred_df = pd.read_csv(predictions_path)
segments_df = pd.read_excel(segments_path)
segments_df.columns = segments_df.columns.str.strip()
pred_df.columns = pred_df.columns.str.strip()

#in case any line from the prediction time is missing information
pred_df['PREDICTION_TIME'] = pd.to_datetime(pred_df['PREDICTION_TIME'], errors='coerce')

#print the size of prediction and segments
print("Predictions shape:", pred_df.shape)
print("Segments shape:", segments_df.shape)



# 3. CLEAN DATA TYPES
# =========================================================

# Convert segment IDs to numeric format for merging
pred_df['SEGMENT_ID'] = pd.to_numeric(pred_df['SEGMENT_ID'], errors='coerce')
segments_df['SEGMENT_ID'] = pd.to_numeric(segments_df['SEGMENT_ID'], errors='coerce')

# Convert coordinate columns to numeric for spatial calculations
coord_cols = [
    'START_LATITUDE', 'START_LONGITUDE',
    'END_LATITUDE', 'END_LONGITUDE'
]

for col in coord_cols:
    segments_df[col] = pd.to_numeric(segments_df[col], errors='coerce')

# Convert predicted speed to numeric for congestion scoring
# Remove rows with missing segment IDs, coordinates, prediction time, or speed
pred_df['Predicted_Speed'] = pd.to_numeric(pred_df['Predicted_Speed'], errors='coerce')

segments_df = segments_df.dropna(subset=['SEGMENT_ID'] + coord_cols).copy()
pred_df = pred_df.dropna(subset=['SEGMENT_ID', 'PREDICTION_TIME', 'Predicted_Speed']).copy()

# Convert segment IDs to integers for consistency across datasets
segments_df['SEGMENT_ID'] = segments_df['SEGMENT_ID'].astype(int)
pred_df['SEGMENT_ID'] = pred_df['SEGMENT_ID'].astype(int)



# 4. MERGE PREDICTIONS WITH SEGMENT COORDINATES
# Combine predicted speeds with segment spatial information using SEGMENT_ID
# Keep only matching records (inner join)
# =========================================================

df = pred_df.merge(
    segments_df,
    on='SEGMENT_ID',
    how='inner'
)

# Print dataset size and number of unique segments after merging
print("Merged shape:", df.shape)
print("Unique mapped segments:", df['SEGMENT_ID'].nunique())



# 5. CREATE SEGMENT MIDPOINTS
# Compute midpoint latitude and longitude for each road segment
# Used for distance calculation and spatial neighbor detection
# =========================================================
segments_df['MID_LAT'] = (
    segments_df['START_LATITUDE'] + segments_df['END_LATITUDE']
) / 2

segments_df['MID_LON'] = (
    segments_df['START_LONGITUDE'] + segments_df['END_LONGITUDE']
) / 2



# 6. DISTANCE FUNCTION
# Computes geographic distance between two latitude/longitude points
# Converts coordinates from degrees to radians
# Uses Haversine formula to account for Earth’s curvature
# Returns distance in kilometers
# Used to identify nearby road segments for spatial analysis
# =========================================================
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c



# 7. BUILD SPATIAL NEIGHBORS
# nearby road segments influence each other
# =========================================================

# Define radius to consider nearby road segments
neighbor_radius_km = 0.45

segment_ids = segments_df['SEGMENT_ID'].tolist()

neighbors = {}

# For each segment, compute distance to all other segments
for seg_id in segment_ids:
    row = segments_df[segments_df['SEGMENT_ID'] == seg_id].iloc[0]

    distances = haversine_distance(
        row['MID_LAT'],
        row['MID_LON'],
        segments_df['MID_LAT'],
        segments_df['MID_LON']
    )

    # Identify segments within the radius (excluding itself)
    # Store nearby segments in a dictionary for each segment
    # Used to capture spatial interaction between roads
    nearby_segments = segments_df.loc[
        (distances <= neighbor_radius_km)
        & (segments_df['SEGMENT_ID'] != seg_id),
        'SEGMENT_ID'
    ].tolist()

    neighbors[seg_id] = nearby_segments

print("Spatial neighbors created")



# 8. CONGESTION SCORE
# lower predicted speed = higher congestion
# =========================================================

# Define reference maximum speed using 95th percentile
max_speed_reference = df['Predicted_Speed'].quantile(0.95)

# Compute congestion score (lower speed → higher congestion)
df['Congestion_Score'] = 1 - (df['Predicted_Speed'] / max_speed_reference)

# Normalize values between 0 and 1 for stability
df['Congestion_Score'] = df['Congestion_Score'].clip(0, 1)



# 9. TEMPORAL CONGESTION
# current and recent congestion are combined
# =========================================================

# Sort data by segment and time to maintain correct order
df = df.sort_values(['SEGMENT_ID', 'PREDICTION_TIME']).reset_index(drop=True)


# Create lagged congestion features (previous time steps)
df['Congestion_Lag_1'] = df.groupby('SEGMENT_ID')['Congestion_Score'].shift(1)
df['Congestion_Lag_2'] = df.groupby('SEGMENT_ID')['Congestion_Score'].shift(2)

# Fill missing lag values with current congestion
df['Congestion_Lag_1'] = df['Congestion_Lag_1'].fillna(df['Congestion_Score'])
df['Congestion_Lag_2'] = df['Congestion_Lag_2'].fillna(df['Congestion_Score'])


# Compute temporal congestion as weighted combination of current and past congestion
df['Temporal_Congestion'] = (
    0.60 * df['Congestion_Score']
    + 0.30 * df['Congestion_Lag_1']
    + 0.10 * df['Congestion_Lag_2']
)


# 10. SPATIAL NEIGHBOR INFLUENCE
# For each segment and time:
# - Retrieve nearby segments using precomputed neighbor dictionary
# - If no neighbors exist, use its own temporal congestion
# - Otherwise, compute average congestion of nearby segments at same time
# - If no nearby data is available at that time, fallback to own congestion
# =========================================================

neighbor_scores = []

for idx, row in df.iterrows():
    seg_id = row['SEGMENT_ID']
    current_time = row['PREDICTION_TIME']

    nearby = neighbors.get(seg_id, [])

    #checks if there are nearby neighbors to use their influence too
    #if not, it only uses its own influnce that is temporal
    if len(nearby) == 0:
        neighbor_scores.append(row['Temporal_Congestion'])
        continue

    nearby_rows = df[
        (df['PREDICTION_TIME'] == current_time)
        & (df['SEGMENT_ID'].isin(nearby))
    ]

    #compute average congestion of nearby segments at same time
    if nearby_rows.empty:
        neighbor_scores.append(row['Temporal_Congestion'])
    else:
        neighbor_scores.append(nearby_rows['Temporal_Congestion'].mean())

# Store result as Neighbor_Congestion to capture spatial influence
df['Neighbor_Congestion'] = neighbor_scores



# 11. FINAL SPATIAL-TEMPORAL PRIORITY
# Combine temporal and spatial congestion into a single score
# Higher weight on temporal congestion (current + recent traffic)
# Lower weight on neighbor congestion (spatial influence)
# =========================================================

df['Priority_Score'] = (
    0.70 * df['Temporal_Congestion']
    + 0.30 * df['Neighbor_Congestion']
)

# Clip values between 0 and 1 for consistency
df['Priority_Score'] = df['Priority_Score'].clip(0, 1)



# 12. CREATE SIGNAL GROUPS BY NEARBY SEGMENTS
# =========================================================

intersection_radius_km = 1.20   # increased so nearby roads are grouped together

# Define radius to group nearby segments into the same intersection
intersection_groups = {}
group_id = 1
assigned_segments = set()


# For each segment:
# - Compute distance to all other segments
# - Group segments within the radius together
# - Assign a unique group ID to each cluster
for seg_id in segment_ids:
    if seg_id in assigned_segments:
        continue

    row = segments_df[segments_df['SEGMENT_ID'] == seg_id].iloc[0]

    distances = haversine_distance(
        row['MID_LAT'],
        row['MID_LON'],
        segments_df['MID_LAT'],
        segments_df['MID_LON']
    )

    # Ensure segments are not assigned to multiple groups
    group_segments = segments_df.loc[
        distances <= intersection_radius_km,
        'SEGMENT_ID'
    ].tolist()

    for s in group_segments:
        intersection_groups[s] = group_id
        assigned_segments.add(s)

    group_id += 1

# Map each segment to its corresponding intersection group
df['Intersection_Group'] = df['SEGMENT_ID'].map(intersection_groups)

# Used to simulate shared traffic signal control
print("Number of simulated signal groups:", df['Intersection_Group'].nunique())
print("Average segments per group:", df.groupby('Intersection_Group')['SEGMENT_ID'].nunique().mean())



# 13. SIGNAL TIMING RULES
# Define minimum and maximum allowed green signal time (in seconds)
# Define total signal cycle time for each intersection
# Initialize list to store optimized results for each group
# =========================================================
MIN_GREEN = 20
MAX_GREEN = 90
CYCLE_TIME = 120

optimized_parts = []



# 14. ALLOCATE GREEN TIME USING SPATIAL-TEMPORAL PRIORITY
# =========================================================


# Process each timestamp and intersection group separately
# Count how many segments exist in each signal group
for (time, group), group_df in df.groupby(['PREDICTION_TIME', 'Intersection_Group']):

    temp = group_df.copy()
    n = len(temp)

    # If only one segment exists:
    # - Assign default base green time
    # - Recommend green time based on its priority score
    if n == 1:
        # single-road group still receives a valid simulated green time
        temp['Base_Green_Time'] = 60
        temp['Recommended_Green_Time'] = (
            MIN_GREEN + temp['Priority_Score'] * (MAX_GREEN - MIN_GREEN)
        )
    # If multiple segments exist:
    # - Divide cycle time equally as base green time
    # - Redistribute cycle time based on priority scores
    # - Higher priority segments receive more green time
    else:
        temp['Base_Green_Time'] = CYCLE_TIME / n

        priority_sum = temp['Priority_Score'].sum()

        # Apply minimum and maximum green time limits
        # Store optimized green time results for all groups
        # Combine all optimized parts into one final dataframe
        if priority_sum == 0:
            temp['Recommended_Green_Time'] = temp['Base_Green_Time']
        else:
            temp['Recommended_Green_Time'] = (
                temp['Priority_Score'] / priority_sum
            ) * CYCLE_TIME

        temp['Recommended_Green_Time'] = temp['Recommended_Green_Time'].clip(
            MIN_GREEN,
            MAX_GREEN
        )

    optimized_parts.append(temp)

optimized_df = pd.concat(optimized_parts, axis=0).reset_index(drop=True)



# 15. SIGNAL PRIORITY + CONGESTION LABELS
# Define priority levels based on priority score thresholds
# Define congestion levels based on predicted speed ranges
# Adds interpretability to optimization results
# =========================================================
def signal_priority_label(score):
    if score >= 0.70:
        return 'High Priority'
    elif score >= 0.40:
        return 'Medium Priority'
    else:
        return 'Low Priority'

def congestion_label(speed):
    if speed < 15:
        return 'Severe Congestion'
    elif speed < 25:
        return 'Moderate Congestion'
    else:
        return 'Free Flow'

# Apply functions to assign:
# - Signal priority label (High, Medium, Low)
# - Traffic congestion level (Severe, Moderate, Free Flow)
optimized_df['Signal_Priority'] = optimized_df['Priority_Score'].apply(signal_priority_label)
optimized_df['Congestion_Level'] = optimized_df['Predicted_Speed'].apply(congestion_label)



# 16. SIMULATED OPTIMIZED SPEED
# =========================================================
optimized_df['Green_Time_Change'] = (
    optimized_df['Recommended_Green_Time'] - optimized_df['Base_Green_Time']
)

# More congested roads benefit more from added green time.
green_gain = np.maximum(optimized_df['Green_Time_Change'], 0)

optimized_df['Estimated_Speed_Improvement'] = (
    green_gain * 0.12 * optimized_df['Priority_Score']
)

# Add a small spatial-temporal benefit for coordinated nearby traffic flow
optimized_df['Coordination_Benefit'] = (
    optimized_df['Neighbor_Congestion'] * 1.25
)

optimized_df['Optimized_Speed'] = (
    optimized_df['Predicted_Speed']
    + optimized_df['Estimated_Speed_Improvement']
    + optimized_df['Coordination_Benefit']
)

# Keep values realistic
optimized_df['Optimized_Speed'] = optimized_df['Optimized_Speed'].clip(
    lower=0,
    upper=max_speed_reference
)



# 17. CHECK DIFFERENCE
# Compute change in speed after optimization
# Calculate average improvement across all segments
# =========================================================
optimized_df['Speed_Change'] = (
    optimized_df['Optimized_Speed'] - optimized_df['Predicted_Speed']
)

# Find maximum improvement achieved
# Count how many segments experienced positive improvement
print("Average speed change:", optimized_df['Speed_Change'].mean())
print("Max speed change:", optimized_df['Speed_Change'].max())
print("Rows improved:", (optimized_df['Speed_Change'] > 0).sum())



# 18. SELECT FINAL OUTPUT COLUMNS
# =========================================================

# Define relevant columns for final analysis and reporting
# Includes prediction, optimization, congestion, priority, signal timing, and spatial data
final_cols = [
    'PREDICTION_TIME',
    'SEGMENT_ID',
    'Intersection_Group',
    'STREET',
    'FROM_STREET',
    'TO_STREET',
    'DIRECTION',
    'Predicted_Speed',
    'Optimized_Speed',
    'Speed_Change',
    'Congestion_Level',
    'Congestion_Score',
    'Temporal_Congestion',
    'Neighbor_Congestion',
    'Priority_Score',
    'Signal_Priority',
    'Base_Green_Time',
    'Recommended_Green_Time',
    'Green_Time_Change',
    'Estimated_Speed_Improvement',
    'Coordination_Benefit',
    'START_LATITUDE',
    'START_LONGITUDE',
    'END_LATITUDE',
    'END_LONGITUDE'
]


# Ensure only existing columns are selected (avoid errors if some are missing)
# Create final cleaned dataset for export or visualization
final_cols = [col for col in final_cols if col in optimized_df.columns]
optimized_output = optimized_df[final_cols].copy()



# 19. SAVE OUTPUT
# =========================================================

#define the path where the output will be saved and save it there
output_path = '/content/drive/MyDrive/datasets/optimized_signal_timing.csv'
optimized_output.to_csv(output_path, index=False)

print("Optimization output saved to:")
print(output_path)

#display the first 20 rows of the output
display(optimized_output.head(20))

Mounted at /content/drive
Predictions shape: (202171, 4)
Segments shape: (55, 11)
Merged shape: (201743, 14)
Unique mapped segments: 55
Spatial neighbors created
Number of simulated signal groups: 28
Average segments per group: 1.9642857142857142
Average speed change: 0.31265467779437434
Max speed change: 5.716214365817137
Rows improved: 191655
Optimization output saved to:
/content/drive/MyDrive/datasets/optimized_signal_timing.csv


,PREDICTION_TIME,SEGMENT_ID,Intersection_Group,STREET,FROM_STREET,TO_STREET,DIRECTION,Predicted_Speed,Optimized_Speed,Speed_Change,...,Signal_Priority,Base_Green_Time,Recommended_Green_Time,Green_Time_Change,Estimated_Speed_Improvement,Coordination_Benefit,START_LATITUDE,START_LONGITUDE,END_LATITUDE,END_LONGITUDE
0,2025-02-07 00:40:00,203,22,North,Pulaski,Kostner,WB,22.111698,22.479169,0.367471,...,Low Priority,60.0,40.578388,-19.421612,0.0,0.367471,41.910024,-87.726329,41.909893,-87.736262
1,2025-02-07 00:50:00,203,22,North,Pulaski,Kostner,WB,25.558376,25.829552,0.271176,...,Low Priority,60.0,35.185846,-24.814154,0.0,0.271176,41.910024,-87.726329,41.909893,-87.736262
2,2025-02-07 01:01:00,203,22,North,Pulaski,Kostner,WB,19.687770,20.072018,0.384248,...,Low Priority,60.0,41.517915,-18.482085,0.0,0.384248,41.910024,-87.726329,41.909893,-87.736262
3,2025-02-07 01:10:00,203,22,North,Pulaski,Kostner,WB,21.298502,21.700714,0.402212,...,Low Priority,60.0,42.523872,-17.476128,0.0,0.402212,41.910024,-87.726329,41.909893,-87.736262
4,2025-02-07 01:20:00,203,22,North,Pulaski,Kostner,WB,23.670150,24.019712,0.349562,...,Low Priority,60.0,39.575460,-20.424540,0.0,0.349562,41.910024,-87.726329,41.909893,-87.736262
5,2025-02-07 01:30:00,203,22,North,Pulaski,Kostner,WB,22.542702,22.884437,0.341735,...,Low Priority,60.0,39.137162,-20.862838,0.0,0.341735,41.910024,-87.726329,41.909893,-87.736262
6,2025-02-07 01:40:00,203,22,North,Pulaski,Kostner,WB,23.194010,23.524182,0.330172,...,Low Priority,60.0,38.489622,-21.510378,0.0,0.330172,41.910024,-87.726329,41.909893,-87.736262
7,2025-02-07 01:50:00,203,22,North,Pulaski,Kostner,WB,22.889593,23.223756,0.334163,...,Low Priority,60.0,38.713137,-21.286863,0.0,0.334163,41.910024,-87.726329,41.909893,-87.736262
8,2025-02-07 02:01:00,203,22,North,Pulaski,Kostner,WB,23.271208,23.597278,0.326070,...,Low Priority,60.0,38.259918,-21.740082,0.0,0.326070,41.910024,-87.726329,41.909893,-87.736262
9,2025-02-07 02:10:00,203,22,North,Pulaski,Kostner,WB,23.387897,23.707818,0.319921,...,Low Priority,60.0,37.915588,-22.084412,0.0,0.319921,41.910024,-87.726329,41.909893,-87.736262


##Visualize both non-optimized and optimized prediction on the map

In [ ]:

# 0. IMPORTS + MOUNT DRIVE
# =========================================================
# pandas and numpy: data loading, cleaning, and calculations
# folium: interactive map visualization
# ipywidgets: dropdowns, selectors, and buttons for interactivity
import pandas as pd
import numpy as np
import folium
import ipywidgets as widgets

# IPython display tools: show and refresh map outputs in Colab
# Google Drive: access saved datasets and output files
# Colab widget manager: enables interactive widgets inside Colab
from IPython.display import display, clear_output, HTML
from google.colab import drive
from google.colab import output as colab_output

# Mount Google Drive and enable widget support
drive.mount('/content/drive')
colab_output.enable_custom_widget_manager()



# 1. LOAD OPTIMIZED OUTPUT
# =========================================================

# Load optimized signal timing dataset from Google Drive
# Clean column names by removing extra spaces
file_path = '/content/drive/MyDrive/datasets/optimized_signal_timing.csv'

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()


# Convert prediction time to datetime format
# Convert coordinate columns to numeric for mapping
df['PREDICTION_TIME'] = pd.to_datetime(df['PREDICTION_TIME'], errors='coerce')

coord_cols = [
    'START_LATITUDE', 'START_LONGITUDE',
    'END_LATITUDE', 'END_LONGITUDE'
]

for col in coord_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')


# Convert key columns (speed, green time, priority) to numeric
# Remove rows with missing critical data (time, speed, coordinates)
df['Predicted_Speed'] = pd.to_numeric(df['Predicted_Speed'], errors='coerce')
df['Optimized_Speed'] = pd.to_numeric(df['Optimized_Speed'], errors='coerce')
df['Recommended_Green_Time'] = pd.to_numeric(df['Recommended_Green_Time'], errors='coerce')
df['Priority_Score'] = pd.to_numeric(df['Priority_Score'], errors='coerce')


# Print dataset size and number of unique times and segments
# Ensures data is clean and ready for visualization
df = df.dropna(subset=[
    'PREDICTION_TIME',
    'SEGMENT_ID',
    'Predicted_Speed',
    'Optimized_Speed',
    'START_LATITUDE',
    'START_LONGITUDE',
    'END_LATITUDE',
    'END_LONGITUDE'
]).copy()

print("Loaded optimized data:", df.shape)
print("Unique times:", df['PREDICTION_TIME'].nunique())
print("Unique segments:", df['SEGMENT_ID'].nunique())



# 2. WIDGET OPTIONS
# =========================================================

# Count available segments for each prediction time
# Create dropdown options showing timestamp and segment count
time_counts = (
    df.groupby('PREDICTION_TIME')['SEGMENT_ID']
    .nunique()
    .reset_index(name='segments_available')
    .sort_values('PREDICTION_TIME')
)

time_options = [
    (f"{row.PREDICTION_TIME} | {row.segments_available} segments", row.PREDICTION_TIME)
    for _, row in time_counts.iterrows()
]


# Reduce huge dropdown
time_options = time_options[::12] if len(time_options) > 150 else time_options


# Reduce dropdown size if there are too many timestamps
# Extract unique segment IDs for multi-selection
segment_options = sorted(df['SEGMENT_ID'].dropna().astype(int).unique().tolist())


# Create widgets:
# - Time dropdown to select prediction timestamp
time_dropdown = widgets.Dropdown(
    options=time_options,
    description='Time:',
    layout=widgets.Layout(width='800px')
)

# - Segment selector to choose road segments
segment_selector = widgets.SelectMultiple(
    options=segment_options,
    value=tuple(segment_options[:10]),
    description='Segments:',
    rows=12,
    layout=widgets.Layout(width='350px')
)

# - Select all button to show all segments
select_all_button = widgets.Button(
    description='Select All',
    button_style='success'
)

# - Clear button to remove selected segments
clear_button = widgets.Button(
    description='Clear',
    button_style='warning'
)

# - Update button to refresh the maps
update_button = widgets.Button(
    description='Update Maps',
    button_style='primary'
)

map_output = widgets.Output()



# 3. COLOR FUNCTION
# Assign color based on traffic speed:
# - Red: severe congestion (slow traffic)
# - Yellow: moderate congestion
# - Green: free-flow traffic
# Used for visual representation on the map
# =========================================================
def speed_color(speed):
    if speed < 15:
        return 'red'
    elif speed < 25:
        return 'yellow'
    else:
        return 'green'



# 4. CREATE SINGLE MAP
# =========================================================

# This function creates an interactive Folium map
# It can visualize either predicted speed or optimized speed
# depending on the selected speed_col parameter
def create_map(filtered, speed_col, title):

    # Calculate the center of the map using the average start/end coordinates
    # This ensures the map opens around the selected road segments
    center_lat = filtered[['START_LATITUDE', 'END_LATITUDE']].mean().mean()
    center_lon = filtered[['START_LONGITUDE', 'END_LONGITUDE']].mean().mean()

    # Create a Folium map using CartoDB Positron tiles
    # CartoDB Positron provides a clean light background for traffic visualization
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='cartodbpositron',
        width='100%',
        height='500px'
    )

    # Initialize bounds list
    # Used later to automatically zoom the map to include all displayed segments
    bounds = []

    # Loop through each road segment in the filtered data
    # Extract start and end coordinates for each segment
    for _, row in filtered.iterrows():

        # Create a popup box for each segment
        # Popup includes:
        # - Segment ID
        # - Street name
        # - From / To streets
        # - Direction
        # - Predicted speed
        # - Optimized speed
        # - Recommended green time
        # - Priority score
        # - Signal priority
        # - Prediction time
        start = [row['START_LATITUDE'], row['START_LONGITUDE']]
        end = [row['END_LATITUDE'], row['END_LONGITUDE']]
        bounds.extend([start, end])

        popup_html = f"""
        <div style="font-family: Arial; font-size: 13px; width: 260px;">
            <h4 style="margin-bottom: 6px;">{title}</h4>
            <b>Segment ID:</b> {row['SEGMENT_ID']}<br>
            <b>Street:</b> {row.get('STREET', '')}<br>
            <b>From:</b> {row.get('FROM_STREET', '')}<br>
            <b>To:</b> {row.get('TO_STREET', '')}<br>
            <b>Direction:</b> {row.get('DIRECTION', '')}<br>
            <hr style="margin: 6px 0;">
            <b>Predicted Speed:</b> {row['Predicted_Speed']:.2f} mph<br>
            <b>Optimized Speed:</b> {row['Optimized_Speed']:.2f} mph<br>
            <b>Recommended Green:</b> {row['Recommended_Green_Time']:.1f} sec<br>
            <b>Priority Score:</b> {row['Priority_Score']:.3f}<br>
            <b>Signal Priority:</b> {row.get('Signal_Priority', '')}<br>
            <b>Time:</b> {row['PREDICTION_TIME']}
        </div>
        """

        # Create tooltip text
        # Tooltip appears when hovering over a road segment
        tooltip_text = (
            f"Segment {row['SEGMENT_ID']} | "
            f"{row.get('STREET', '')} | "
            f"{speed_col}: {row[speed_col]:.1f} mph"
        )

        # Draw each road segment as a PolyLine
        # PolyLine connects start and end coordinates
        # Line color is based on the selected speed column:
        # - red = slow
        # - yellow = medium
        # - green = fast
        # Add each PolyLine to the map
        folium.PolyLine(
            locations=[start, end],
            color=speed_color(row[speed_col]),
            weight=9,
            opacity=0.9,
            popup=folium.Popup(popup_html, max_width=320),
            tooltip=tooltip_text
        ).add_to(m)

    # Fit map bounds to displayed road segments
    # This automatically adjusts zoom and position
    if bounds:
        m.fit_bounds(bounds)

    # Add a custom legend to explain speed colors
    legend_html = """
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        width: 190px;
        height: 115px;
        background-color: white;
        border:2px solid grey;
        z-index:9999;
        font-size:14px;
        padding: 10px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    ">
    <b>Speed Level</b><br>
    <span style="color:red;">●</span> Slow (&lt; 15 mph)<br>
    <span style="color:gold;">●</span> Medium (15–25 mph)<br>
    <span style="color:green;">●</span> Fast (≥ 25 mph)
    </div>
    """

    m.get_root().html.add_child(folium.Element(legend_html))
    return m



# 5. DRAW SIDE-BY-SIDE MAPS
# =========================================================

# Filter optimized data by selected prediction time
# Further filter by selected road segments if chosen
def draw_comparison_maps(selected_time, selected_segments):
    filtered = df[df['PREDICTION_TIME'] == selected_time].copy()

    if selected_segments:
        filtered = filtered[filtered['SEGMENT_ID'].isin(list(selected_segments))].copy()

    # Print selected time and number of displayed segments
    # Stop if no data is available for the selected filters
    print("Selected time:", selected_time)
    print("Segments shown:", filtered['SEGMENT_ID'].nunique())

    # Create first map using Predicted_Speed
    # This represents traffic conditions before optimization
    if filtered.empty:
        print("No data available for the selected time/segments.")
        return

    before_map = create_map(
        filtered,
        speed_col='Predicted_Speed',
        title='Before Optimization'
    )

    # Create second map using Optimized_Speed
    # This represents traffic conditions after optimization
    after_map = create_map(
        filtered,
        speed_col='Optimized_Speed',
        title='After Optimization'
    )

    # Convert both Folium maps to HTML
    before_html = before_map._repr_html_()
    after_html = after_map._repr_html_()


    # Display them side-by-side for easy visual comparison
    comparison_html = f"""
    <div style="display:flex; gap:10px; width:100%;">
        <div style="width:50%;">
            <h3 style="text-align:center;">Before Optimization</h3>
            {before_html}
        </div>
        <div style="width:50%;">
            <h3 style="text-align:center;">After Optimization</h3>
            {after_html}
        </div>
    </div>
    """

    display(HTML(comparison_html))

    # Compute average values for the selected view:
    # - Predicted speed
    # - Optimized speed
    # - Recommended green time
    # - Priority score
    summary = filtered[['Predicted_Speed', 'Optimized_Speed', 'Recommended_Green_Time', 'Priority_Score']].mean()

    print("\nAverage values for selected view")
    print("--------------------------------")
    print(f"Average predicted speed : {summary['Predicted_Speed']:.2f} mph")
    print(f"Average optimized speed : {summary['Optimized_Speed']:.2f} mph")
    print(f"Average green time      : {summary['Recommended_Green_Time']:.2f} sec")
    print(f"Average priority score  : {summary['Priority_Score']:.3f}")



# 6. BUTTON ACTIONS
# =========================================================

# Define behavior when "Update Maps" button is clicked:
# - Clear previous map output
# - Generate updated comparison maps based on selected time and segments
def update_maps_clicked(button):
    with map_output:
        clear_output(wait=True)
        draw_comparison_maps(
            time_dropdown.value,
            segment_selector.value
        )

# Define behavior for "Select All" button:
# - Select all available segments in the selector
def select_all_clicked(button):
    segment_selector.value = tuple(segment_options)


# Define behavior for "Clear" button:
# - Deselect all segments
def clear_clicked(button):
    segment_selector.value = tuple()



# Link buttons to their respective functions
# Enables interactive control of the visualization
update_button.on_click(update_maps_clicked)
select_all_button.on_click(select_all_clicked)
clear_button.on_click(clear_clicked)



# 7. DISPLAY CONTROLS
# =========================================================

# Arrange UI elements:
# - Time dropdown on top
# - Segment selector with action buttons beside it
controls = widgets.VBox([
    time_dropdown,
    widgets.HBox([
        segment_selector,
        widgets.VBox([
            select_all_button,
            clear_button,
            update_button
        ])
    ])
])

# Display control panel (filters and buttons)
# Display output area where maps will be rendered
display(controls)
display(map_output)

with map_output:
    draw_comparison_maps(time_dropdown.value, segment_selector.value)

Mounted at /content/drive
Loaded optimized data: (201743, 25)
Unique times: 4901
Unique segments: 55


Output()